# Extraction du NDVI par point GPS — Google Earth Engine
## Mémoire AgriSmart — Enrichissement satellitaire du dataset Sénégal

**Objectif :** ajouter à chaque point GPS du dataset deux indicateurs de télédétection :
- `ndvi_moyen_annuel` : vigueur végétale moyenne (2019-2024)
- `ndvi_hivernage` : vigueur végétale pendant la saison des pluies (juin-octobre)

**Source :** MODIS MOD13Q1 (NDVI, résolution 250 m, composite 16 jours) — même
résolution que SoilGrids, cohérent avec le reste du dataset.

**Mode d'emploi (comme tes notebooks lumières nocturnes) :**
1. Exécute les cellules dans l'ordre
2. À la cellule 2, authentifie-toi avec ton compte Google Earth Engine
3. À la cellule 3, téléverse ton fichier `dataset_senegal_brut.csv`
4. À la fin, le fichier enrichi se télécharge automatiquement

## 1. Installation et authentification Earth Engine

In [ ]:
import ee

# Authentification (une fenêtre s'ouvre — connecte ton compte GEE)
ee.Authenticate()

# ⚠️ Remplace par TON project ID Google Cloud (le même que pour
# les lumières nocturnes, visible sur code.earthengine.google.com)
ee.Initialize(project='wise-obelisk-501712-b5')
print("Earth Engine initialisé ✅")

Earth Engine initialisé ✅


## 2. Téléversement du dataset

In [ ]:
from google.colab import files
import pandas as pd
import io

print("Téléverse dataset_senegal_brut.csv :")
up = files.upload()
df = pd.read_csv(io.BytesIO(list(up.values())[0]))

# Points uniques (le brut a 1 ligne par point, mais on sécurise)
pts = df[["latitude", "longitude"]].drop_duplicates().reset_index(drop=True)
print(f"{len(pts)} points GPS à traiter")

Téléverse dataset_senegal_brut.csv :


Saving dataset_senegal_brut.csv to dataset_senegal_brut.csv
900 points GPS à traiter


## 3. Préparation des composites NDVI

MOD13Q1 fournit le NDVI en valeurs entières (facteur d'échelle 0,0001).
On construit deux images : moyenne annuelle et moyenne de l'hivernage,
sur la période 2019-2024 (6 ans lissent les années exceptionnelles).

In [ ]:
DEBUT, FIN = "2019-01-01", "2024-12-31"

modis = (ee.ImageCollection("MODIS/061/MOD13Q1")
           .filterDate(DEBUT, FIN)
           .select("NDVI"))

# NDVI moyen annuel (toutes images)
ndvi_annuel = modis.mean().multiply(0.0001).rename("ndvi_moyen_annuel")

# NDVI moyen de l'hivernage (juin -> octobre uniquement)
ndvi_hiv = (modis.filter(ee.Filter.calendarRange(6, 10, "month"))
                 .mean().multiply(0.0001).rename("ndvi_hivernage"))

image = ndvi_annuel.addBands(ndvi_hiv)
print("Composites NDVI prêts ✅")

Composites NDVI prêts ✅


## 4. Échantillonnage aux points GPS

On interroge l'image par lots de 100 points (limite de sécurité de l'API
`getInfo`), avec une petite zone tampon implicite : l'échelle de 250 m
correspond au pixel MODIS.

In [ ]:
resultats = []
LOT = 100

for debut in range(0, len(pts), LOT):
    lot = pts.iloc[debut:debut + LOT]
    fc = ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([r.longitude, r.latitude]),
                   {"latitude": r.latitude, "longitude": r.longitude})
        for r in lot.itertuples()
    ])
    ech = image.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.mean(),
        scale=250,
    ).getInfo()
    for f in ech["features"]:
        p = f["properties"]
        resultats.append({
            "latitude": p["latitude"],
            "longitude": p["longitude"],
            "ndvi_moyen_annuel": p.get("ndvi_moyen_annuel"),
            "ndvi_hivernage": p.get("ndvi_hivernage"),
        })
    print(f"  {min(debut + LOT, len(pts))}/{len(pts)} points traités")

ndvi = pd.DataFrame(resultats).round(4)
print("\nAperçu :")
ndvi.head()

  100/900 points traités
  200/900 points traités
  300/900 points traités
  400/900 points traités
  500/900 points traités
  600/900 points traités
  700/900 points traités
  800/900 points traités
  900/900 points traités

Aperçu :


,latitude,longitude,ndvi_moyen_annuel,ndvi_hivernage
0,15.6391,-14.8962,0.2886,0.3843
1,16.6009,-13.4075,0.2009,0.2480
2,16.0804,-14.1768,0.2410,0.3267
3,15.6255,-16.3968,0.2285,0.2810
4,16.0702,-16.1445,0.2132,0.2352


## 5. Contrôle qualité rapide

In [ ]:
print("Valeurs manquantes :")
print(ndvi.isna().sum())
print(f"\nNDVI annuel  : min={ndvi.ndvi_moyen_annuel.min():.3f}  "
      f"max={ndvi.ndvi_moyen_annuel.max():.3f}")
print(f"NDVI hivernage : min={ndvi.ndvi_hivernage.min():.3f}  "
      f"max={ndvi.ndvi_hivernage.max():.3f}")

# Cohérence attendue : hivernage > annuel (végétation de saison des pluies),
# Casamance > zone sylvopastorale, valeurs entre ~0.1 (sahel) et ~0.7 (forêt)
verif = df.merge(ndvi, on=["latitude", "longitude"], how="left")
print("\nNDVI hivernage moyen par zone (contrôle de cohérence) :")
print(verif.groupby("zone_agroecologique")["ndvi_hivernage"]
      .mean().round(3).sort_values(ascending=False))

Valeurs manquantes :
latitude              0
longitude             0
ndvi_moyen_annuel    30
ndvi_hivernage       30
dtype: int64

NDVI annuel  : min=-0.069  max=0.730
NDVI hivernage : min=-0.040  max=0.742

NDVI hivernage moyen par zone (contrôle de cohérence) :
zone_agroecologique
Casamance              0.623
Senegal_Oriental       0.613
Bassin_Arachidier      0.430
Zone_Sylvopastorale    0.375
Niayes                 0.356
Vallee_du_Fleuve       0.279
Name: ndvi_hivernage, dtype: float64


## 6. Fusion avec le dataset et téléchargement

In [ ]:
final = df.merge(ndvi, on=["latitude", "longitude"], how="left")
final.to_csv("dataset_senegal_brut_ndvi.csv", index=False,
             encoding="utf-8-sig")
print(f"Fichier fusionné : {final.shape[0]} lignes x {final.shape[1]} colonnes")
files.download("dataset_senegal_brut_ndvi.csv")

Fichier fusionné : 900 lignes x 20 colonnes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Étapes suivantes (hors Colab)

1. Remplace `dataset_senegal_brut.csv` par `dataset_senegal_brut_ndvi.csv`
   dans ton dossier `memoire`.
2. Dans **construire_etiquettes.py**, ajoute les deux colonnes à la liste
   `variables` :
   ```python
   variables = [..., "pluie_hivernage_mm", "ndvi_moyen_annuel", "ndvi_hivernage"]
   ```
3. Relance l'étiquetage :
   `python construire_etiquettes.py --entree dataset_senegal_brut_ndvi.csv`
4. Dans le **notebook d'entraînement**, ajoute les deux colonnes à `FEATURES`
   et ré-exécute : le graphique d'importance des variables te dira ce que
   le NDVI apporte au modèle — un résultat à commenter dans le mémoire.

**Note pour le mémoire :** le NDVI n'intervient pas dans les règles
d'étiquetage Ecocrop (qui restent fondées sur température, pluie et pH) ;
il enrichit uniquement les variables d'entrée. Le modèle peut ainsi
exploiter une information de terrain observée par satellite, indépendante
des règles — un premier pas vers l'ancrage dans des données réelles.